# DroneFuse - Step 2b: drone-centred camera crops (GPU)

The whole-frame camera embeddings of Step 2 reach 100 % on validation, which a drone a few pixels wide
cannot explain: they most likely recognise the recording (clouds, light, site). This notebook finds the
drone in each frame without labels and embeds only a small crop around it:
- black-hat filter on the brightest colour channel (the drone is dark in every channel, blue sky is not);
- minus what is always there (median over the recording, sky mask) and what barely moves (the same spot
  0.3 s earlier and later, e.g. gaps between clouds);
- per window, the 3 frames with the strongest speck; crops standardised per colour channel; ConvNeXt-T.

It also runs two checks on train/val only (test is not used):
- the closed-set linear probe again, now including the crops;
- a **recording probe**: can a sensor tell flight 1 from flight 2 of the same drone type at the car park?
  Chance is 50 %. A high score means the features identify the recording.

**Kaggle settings:** Accelerator = **GPU T4**. Internet = **On** (the ConvNeXt weights are downloaded).
**Inputs to add:** `dronefuse-code` (latest), the three Step 1 datasets `dronefuse-mmaud-v1`, `-v2a`, `-v2b`,
and `dronefuse-step2-features`.
**Output:** `<recording>.crop.npz` and a contact sheet `<recording>_crops.jpg` per recording, `step2b_report.txt`.
**Expected run time:** about 1 GPU-hour.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step2b')
ONLY_RECORDINGS = None            # e.g. ['V2_p4s1'] to process one recording
TIME_BUDGET_MINUTES = 11 * 60

In [ ]:
import json

from dronefuse import environment
from dronefuse.runner import StepRunner
from dronefuse.step2 import find_recordings
from dronefuse.windows import load_windows

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'step2b_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
windows = load_windows(code_root / 'splits' / 'windows.csv')
deep_files = sorted(INPUT.rglob('*.deep.npz'))
if not deep_files:
    raise RuntimeError('No Step 2 features found. Add Input -> Your Datasets -> dronefuse-step2-features, '
                       'then Save & Run All again.')
STEP2 = deep_files[0].parent
recordings = find_recordings(INPUT)
if ONLY_RECORDINGS:
    recordings = {k: v for k, v in recordings.items() if k in ONLY_RECORDINGS}
print('Step 2 features:', STEP2)
print('recordings found:', {k: str(v) for k, v in recordings.items()})
missing = sorted({w['recording'] for w in windows} - set(recordings))
print('not attached (skipped):', missing)
summary['recordings_found'] = {k: str(v) for k, v in recordings.items()}
save_summary()

In [ ]:
# The crops: one step per recording (a few minutes each).
from dronefuse.crops import recording_crops

crops = {}
for name, folder in recordings.items():
    crops[name] = runner.run(f'crops {name}', recording_crops, name, folder, STEP2, OUTPUT)
    summary['crops'] = crops
    save_summary()

In [ ]:
# Checks on train/val only: closed-set probe (with crops) and the recording probe.
from dronefuse.step2 import linear_probe, recording_probe

done = [n for n, r in crops.items() if r is not None]
probe = runner.run('linear probe (val only)', linear_probe, STEP2, windows, done, print, OUTPUT)
flights = runner.run('recording probe (val only)', recording_probe, STEP2, windows, done, OUTPUT)
summary['probe'], summary['recording_probe'] = probe, flights

lines = ['CROPS (peak = strength of the speck in the best frame; percentiles 10/50/90 over windows)']
for name, r in crops.items():
    lines.append(f'  {name}: ' + ('FAILED, see the errors above' if r is None else
                                  f"{r['windows']} windows, peak {r['peak_q10_50_90']}, sheet {r['sheet']}"))
lines.append('\nLINEAR PROBE, closed protocol, train -> VAL (drone type; test not used)')
for block, r in (probe or {}).items():
    lines.append(f'  {block:24s} ' + ('not available' if not r or 'note' in r else
                                       f"acc {r['val_accuracy']:.3f}  balanced {r['val_balanced_accuracy']:.3f}"))
lines.append('\nRECORDING PROBE, car park: flight 1 vs flight 2 of the same type, train -> VAL (chance 0.500)')
for block, r in (flights or {}).items():
    lines.append(f'  {block:24s} ' + ('not available' if not r else
                                       f"balanced acc {r['balanced_accuracy']:.3f}  per type (M300, M30, P4) {r['per_type']}"))
failed = runner.failed()
lines.append('\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed)))
lines.append('RECORDINGS NOT ATTACHED: ' + (', '.join(missing) or 'none'))
saved = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
lines.append(f'SAVED: {saved / 1e9:.2f} GB in {OUTPUT}')
report = '\n'.join(lines)
(OUTPUT / 'step2b_report.txt').write_text(report)
save_summary()
print(report)

In [ ]:
# A look at the crops of one recording (the contact sheets are also saved in the output).
from IPython.display import Image as Show, display

for sheet in sorted(OUTPUT.glob('*_crops.jpg'))[:2]:
    print(sheet.name)
    display(Show(filename=str(sheet)))

## After the run

1. Send back `dronefuse_outputs/step2b/step2b_report.txt`, plus two or three of the `*_crops.jpg` sheets
   (one from V1, one from V2) so we can check by eye that the crops contain the drone.
2. Open this notebook's saved version -> **Output** -> **New Dataset**, name it **`dronefuse-step2b-crops`**
   (private). Step 3 uses it for the drone-centred camera variant.